# DVC — Data Version Control for ML

DVC is Git for data and ML pipelines. It tracks datasets, model files, and experiment pipelines with the same workflows as Git.

**Topics:** DVC init, tracking data, pipeline stages, remote storage, experiment comparison

In [ ]:
# Install: pip install dvc dvc-s3 dvc-gcs dvc-azure
# DVC requires git repository

dvc_setup = """
=== DVC Initial Setup ===

# 1. Initialize DVC in a git repo
git init
dvc init
git add .dvc/ .dvcignore
git commit -m "chore: initialize DVC"

# 2. Configure remote storage (where large files actually live)
# S3
dvc remote add -d myremote s3://my-ml-bucket/dvc
dvc remote modify myremote region us-east-1

# GCS
dvc remote add -d myremote gs://my-ml-bucket/dvc

# Local (for testing)
dvc remote add -d localremote /tmp/dvc-storage

git add .dvc/config
git commit -m "chore: configure DVC remote storage"
"""
print(dvc_setup)

## 1. Tracking Data Files

In [ ]:
data_tracking = """
=== Tracking Data with DVC ===

# Add large data files to DVC (not git!)
dvc add data/raw/training_data.csv       # creates data/raw/training_data.csv.dvc
dvc add data/raw/features_store.parquet
dvc add models/production_model_v2.pkl

# .dvc file is small and goes in git
git add data/raw/training_data.csv.dvc   # add the pointer
git add data/.gitignore                  # DVC auto-adds to .gitignore
git commit -m "data: add Q4 2024 training dataset (500K rows)"

# Push data to remote storage
dvc push    # uploads to S3/GCS/etc.

# On another machine / in CI:
git pull                     # get .dvc pointer files
dvc pull                     # download actual data from remote

# Update data (new version)
# ... update the CSV file ...
dvc add data/raw/training_data.csv       # re-adds (updates .dvc file)
git add data/raw/training_data.csv.dvc
git commit -m "data: add Q1 2025 data (+50K new samples)"
dvc push

# Checkout old version
git checkout HEAD~1 -- data/raw/training_data.csv.dvc
dvc checkout              # restores old data file
"""
print(data_tracking)

## 2. DVC Pipeline — Reproducible ML

In [ ]:
dvc_yaml = """
# dvc.yaml — define ML pipeline stages
stages:
  data_prep:
    cmd: python src/data_prep.py
    deps:
      - src/data_prep.py
      - data/raw/training_data.csv
    params:
      - params.yaml:
        - data_prep.test_size
        - data_prep.random_state
    outs:
      - data/processed/train.parquet
      - data/processed/test.parquet

  feature_engineering:
    cmd: python src/features.py
    deps:
      - src/features.py
      - data/processed/train.parquet
    params:
      - params.yaml:
        - features.polynomial_degree
        - features.interaction_terms
    outs:
      - data/features/train_features.parquet
      - data/features/feature_schema.json

  train:
    cmd: python src/train.py
    deps:
      - src/train.py
      - data/features/train_features.parquet
    params:
      - params.yaml:
        - train.n_estimators
        - train.max_depth
        - train.learning_rate
    outs:
      - models/model.pkl
    metrics:
      - metrics/train_metrics.json:  # tracked metrics
          cache: false

  evaluate:
    cmd: python src/evaluate.py
    deps:
      - src/evaluate.py
      - models/model.pkl
      - data/processed/test.parquet
    metrics:
      - metrics/eval_metrics.json:
          cache: false
    plots:
      - plots/roc_curve.csv:
          x: fpr
          y: tpr
"""

params_yaml = """
# params.yaml — all hyperparameters in one place
data_prep:
  test_size: 0.2
  random_state: 42

features:
  polynomial_degree: 2
  interaction_terms: true

train:
  n_estimators: 100
  max_depth: 4
  learning_rate: 0.1
  subsample: 0.8
"""

print('=== dvc.yaml (pipeline definition) ===')
print(dvc_yaml)
print('=== params.yaml ===')
print(params_yaml)

pipeline_commands = """
Pipeline commands:
  dvc repro                # run all changed stages
  dvc repro train          # run from 'train' stage only
  dvc dag                  # visualize pipeline DAG
  dvc params diff          # show param changes
  dvc metrics show         # show current metrics
  dvc metrics diff         # compare metrics with git HEAD
  dvc plots show           # generate HTML plots

Experiment comparison:
  dvc exp run -S train.learning_rate=0.05    # run with different param
  dvc exp run -S train.n_estimators=200
  dvc exp show                               # table comparison of all runs
  dvc exp diff exp_a exp_b                   # diff two experiments
  dvc exp apply best_experiment_id           # apply best experiment params
"""
print(pipeline_commands)

## 3. DVC + GitHub Actions CI

In [ ]:
ci_workflow = """
# .github/workflows/dvc-pipeline.yml
name: DVC ML Pipeline

on:
  push:
    branches: [main]
  pull_request:
    paths:
      - 'src/**'
      - 'params.yaml'
      - 'dvc.yaml'

jobs:
  ml-pipeline:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v4
      
      - uses: iterative/setup-dvc@v1
      
      - name: Configure DVC remote credentials
        run: |
          dvc remote modify myremote access_key_id ${{ secrets.AWS_ACCESS_KEY_ID }}
          dvc remote modify myremote secret_access_key ${{ secrets.AWS_SECRET_ACCESS_KEY }}
      
      - name: Pull DVC data
        run: dvc pull data/raw/ data/processed/
      
      - name: Run pipeline (only changed stages)
        run: dvc repro
      
      - name: Check metrics threshold
        run: |
          AUC=$(python -c "import json; print(json.load(open('metrics/eval_metrics.json'))['auc'])")
          python -c "assert float('$AUC') >= 0.90, f'AUC {$AUC} below threshold 0.90'"
      
      - name: Push results
        run: dvc push
      
      - name: Report metrics in PR
        uses: iterative/cml@v2  # CML: Continuous Machine Learning
        with:
          token: ${{ secrets.GITHUB_TOKEN }}
          command: |
            echo '## Model Evaluation' > report.md
            dvc metrics show --md >> report.md
            dvc plots diff HEAD main --target plots/ --show-vega | cml send-comment report.md
"""
print('=== DVC + GitHub Actions CI ===')
print(ci_workflow)

print("""Key DVC benefits for ML teams:
  1. Data versioning without storing large files in Git
  2. Reproducible pipelines — dvc repro re-runs only changed stages
  3. Experiment tracking with parameter sweeps (dvc exp run)
  4. Works with any remote: S3, GCS, Azure, HDFS, SSH, local
  5. CI/CD integration for automated model evaluation
  6. CML (Continuous ML) for PR comments with metric tables + plots
""")